# Topic 5: Sorting Algorithms

## 5.1 Why Learn Sorting Algorithms?

Sorting teaches **divide-and-conquer thinking, comparison logic, stability, and algorithm analysis** — all transferable to ML model selection, data preprocessing, and interviews.

| Algorithm | Best | Average | Worst | Space | Stable | Use When |
|-----------|------|---------|-------|-------|--------|----------|
| Bubble Sort | O(n) | O(n²) | O(n²) | O(1) | ✅ | Educational only |
| Selection Sort | O(n²) | O(n²) | O(n²) | O(1) | ❌ | Small data, minimise swaps |
| Insertion Sort | O(n) | O(n²) | O(n²) | O(1) | ✅ | Nearly sorted data, small arrays |
| Merge Sort | O(n log n) | O(n log n) | O(n log n) | O(n) | ✅ | Need guaranteed O(n log n), stability |
| Quick Sort | O(n log n) | O(n log n) | O(n²) | O(log n) | ❌ | General purpose, cache-friendly |
| Heap Sort | O(n log n) | O(n log n) | O(n log n) | O(1) | ❌ | O(1) space + O(n log n) guarantee |
| **Timsort (Python)** | **O(n)** | **O(n log n)** | **O(n log n)** | **O(n)** | ✅ | Python's `sorted()` — gold standard |

---

## 5.2 Implementations — From Naive to Efficient

In [ ]:
# ── BUBBLE SORT — O(n²) — educational only ───────────────────────────────────
def bubble_sort(arr):
    arr = arr[:]   # don't modify original
    n = len(arr)
    for i in range(n):
        swapped = False
        for j in range(0, n-i-1):       # last i elements already sorted
            if arr[j] > arr[j+1]:
                arr[j], arr[j+1] = arr[j+1], arr[j]
                swapped = True
        if not swapped: break           # O(n) best case: already sorted
    return arr


# ── INSERTION SORT — O(n²) avg, O(n) best — great for small/nearly sorted ────
def insertion_sort(arr):
    arr = arr[:]
    for i in range(1, len(arr)):
        key = arr[i]
        j   = i - 1
        while j >= 0 and arr[j] > key:  # shift larger elements right
            arr[j+1] = arr[j]
            j -= 1
        arr[j+1] = key                  # insert key in correct position
    return arr


# ── MERGE SORT — O(n log n) guaranteed — stable ───────────────────────────────
def merge_sort(arr):
    if len(arr) <= 1: return arr
    mid   = len(arr) // 2
    left  = merge_sort(arr[:mid])
    right = merge_sort(arr[mid:])
    result = []
    i = j  = 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]: result.append(left[i]);  i += 1
        else:                   result.append(right[j]); j += 1
    return result + left[i:] + right[j:]


# ── QUICK SORT — O(n log n) avg — in-place ────────────────────────────────────
def quick_sort(arr, low=0, high=None):
    if high is None: high = len(arr) - 1
    if low < high:
        pi = _partition(arr, low, high)
        quick_sort(arr, low, pi - 1)
        quick_sort(arr, pi + 1, high)
    return arr

def _partition(arr, low, high):
    pivot = arr[high]           # last element as pivot
    i     = low - 1
    for j in range(low, high):
        if arr[j] <= pivot:
            i += 1
            arr[i], arr[j] = arr[j], arr[i]
    arr[i+1], arr[high] = arr[high], arr[i+1]
    return i + 1


# ── Compare all on same data ──────────────────────────────────────────────────
import time
import random

data = [random.randint(1, 1000) for _ in range(2000)]

def bench(name, func, arr):
    start = time.perf_counter()
    result = func(arr[:])
    elapsed = time.perf_counter() - start
    print(f"  {name:20s}: {elapsed*1000:.2f}ms  first5={result[:5]}")

print("Sorting 2000 random integers:")
bench("bubble_sort",    bubble_sort,    data)
bench("insertion_sort", insertion_sort, data)
bench("merge_sort",     merge_sort,     data)
bench("quick_sort",     lambda a: quick_sort(a[:]), data)
bench("Python sorted()",sorted,         data)

## 5.3 Sorting in Python and AI/ML Context

> 🔵 **[AI/ML]** Timsort performs O(n) on already-sorted data — this is why pre-sorting training data sometimes speeds up batch generation. Top-K selection using sort is O(n log n) — using a heap is **O(n log k)** which is MUCH faster when `k << n`. This is how recommendation systems return top-K items.

In [ ]:
# ── Python's built-in sort (Timsort — O(n log n), stable) ───────────────────
nums = [3, 1, 4, 1, 5, 9, 2, 6]
print(sorted(nums))              # new sorted list
print(sorted(nums, reverse=True))

# Sort complex objects with key=
students = [
    {'name': 'Carol', 'gpa': 9.1},
    {'name': 'Alice', 'gpa': 8.5},
    {'name': 'Bob',   'gpa': 7.2},
]
by_gpa = sorted(students, key=lambda s: s['gpa'], reverse=True)
print("Top student:", by_gpa[0]['name'])   # Carol

# Multi-key sort: primary by dept, secondary by salary desc
employees = [
    {'name':'Alice', 'dept':'Eng', 'salary':90000},
    {'name':'Bob',   'dept':'Mkt', 'salary':70000},
    {'name':'Carol', 'dept':'Eng', 'salary':85000},
]
sorted_emp = sorted(employees, key=lambda e: (e['dept'], -e['salary']))
for e in sorted_emp:
    print(f"  {e['dept']:4} | {e['name']:6} | {e['salary']:,}")

# ── AI/ML: top-K predictions by confidence ───────────────────────────────────
import heapq
predictions = [
    {'label': 'cat',  'confidence': 0.72},
    {'label': 'dog',  'confidence': 0.91},
    {'label': 'bird', 'confidence': 0.61},
    {'label': 'fish', 'confidence': 0.45},
]
# O(n log k) — much better than full sort for large n, small k
top2 = heapq.nlargest(2, predictions, key=lambda p: p['confidence'])
print("\nTop 2:", [(p['label'], p['confidence']) for p in top2])

## ✏️ Exercises — Sorting

**[EXERCISE 5.1 — Medium]** Implement `counting_sort(arr)` for non-negative integers. Count occurrences, then reconstruct. Time: O(n+k), Space: O(k). Test with `[4,2,2,8,3,3,1]`.

**[EXERCISE 5.2 — Challenge]** Write `kth_largest(arr, k)` that returns the k-th largest element in O(n) average time using **QuickSelect** (same partition idea as QuickSort but only recurse into one half).